In [ ]:
# Finalspark header
import numpy as np
import time
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output
from neuroplatform import StimShape, StimParam , IntanSofware, Trigger, Database, StimPolarity, Experiment

import pandas as pd
from dateutil import parser
from lets_plot import *
from tqdm import tqdm
LetsPlot.setup_html()

In [ ]:
# Establish API credentials
token = "<insert token here>"
exp = Experiment(token)
print(f'Electrodes: {exp.electrodes}') # Electrodes that you can used


## DEPRECATED protocol for Iteration 4: Phase shift induction.

In [ ]:
# Basic Stim obj 1
stim_paramaters, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
stim_paramaters = StimParam()
stim_paramaters.index = exp.electrodes[8]
stim_paramaters.trigger_key = 0
stim_paramaters.phase_amplitude1 = 10
stim_paramaters.phase_amplitude2 = 10
stim_paramaters.display_attributes()

print(last_updated_paramaters)

In [ ]:
stim_paramaters2, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
stim_paramaters2 = StimParam()
stim_paramaters2.index = exp.electrodes[9]
stim_paramaters2.trigger_key = 1
stim_paramaters2.phase_amplitude1 = 10
stim_paramaters2.phase_amplitude2 = 10
stim_paramaters2.display_attributes()

print(last_updated_paramaters)

In [ ]:
stim_paramaters3, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
stim_paramaters3 = StimParam()
stim_paramaters3.index = exp.electrodes[5]
stim_paramaters3.trigger_key = 2
stim_paramaters3.phase_amplitude1 = 10
stim_paramaters3.phase_amplitude2 = 10
stim_paramaters3.display_attributes()

print(last_updated_paramaters)

In [ ]:
# Natural frequencies derived from FFT baseline
phase1Hz = 8
phase1Sleep = 1/phase1Hz

phase2Hz = 4
phase2Sleep = 1/phase2Hz

intan = IntanSofware()
trigger_gen = Trigger()

stim_param_list = [stim_paramaters, stim_paramaters2]
phaseShiftSwitchTimestamps = []

intan.set_count([0])

try:
    if exp.start(): # Signal the start of an experiment to all users
        # Measure impedance
        intan.impedance()

        # Disable Variation STD (keep a fixed threshold)
        intan.var_threshold(False)

        # Send stim parameter
        intan.send_stimparam(stim_param_list)
        start_exp = datetime.utcnow()

        phase1end = start_exp + timedelta(minutes=5)
        phase2end = start_exp + timedelta(minutes=10)

        # Trigger 0 only = electrode 0
        trigger0 = np.zeros(16, dtype=np.uint8)
        trigger1 = np.zeros(16, dtype=np.uint8)
        trigger2 = np.zeros(16, dtype=np.uint8)
        trigger0[0] = 1
        trigger1[1] = 1
        trigger2[2] = 1

        while datetime.utcnow() < phase1end:
            trigger_gen.send(trigger0)
            time.sleep(0.005)
            trigger_gen.send(trigger2)
            time.sleep(phase1Sleep-0.005)

        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        while datetime.utcnow() < phase2end:
            trigger_gen.send(trigger1)
            time.sleep(phase2Sleep)

        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # Disable all stims
        for stim in stim_param_list:
            stim.enable = False
        intan.send_stimparam(stim_param_list)

        stop_exp = datetime.utcnow()

finally:
    # Close the connection to trigger generator
    trigger_gen.close()
    # Enable variation threshold again
    intan.var_threshold(True)
    # Close the connection to intan software
    intan.close()
    # Signal the end of an experiment to all users
    exp.stop()


In [ ]:
print(phaseShiftSwitchTimestamps)

# Timestamps
print(start_exp)

# Timestamps
print(stop_exp)

In [ ]:
# Establish db connection and get exp name
db = Database()
exp_name = exp.exp_name
exp_name

In [ ]:
# Get current time func
def get_current_utc_time():
    """Returns the current UTC time as a datetime object."""
    return datetime.now(timezone.utc)

In [ ]:
# Start time parser - Timestamps gained from experiment protocols
s1 = datetime(2024, 5, 2, 9, 0, 0, 941232)
s1

# End time parser
s2 = datetime(2024, 5, 2, 12, 0, 0, 956840)
s2

In [ ]:
# Trigger data collection point
triggers = db.get_all_triggers(s1, s2)
triggers

In [ ]:
# SPM data collection point
df_spike_count = db.get_spike_count(s1, s2, exp_name)
df_spike_count

In [ ]:
# Raw spike data collection point
df_spike_event = db.get_spike_event(s1,s2,exp_name)
df_spike_event

## Below are several plotting features that can be run. For large datasets batch processing is required for effective raw-data analysis


In [ ]:
# Plot SPM
def plot_df(df: pd.DataFrame, title: str):
    return ggplot(df, aes(x='Time', y='Spike per minutes', group='channel', color='channel')) + ggsize(800, 600) + geom_line() + \
    ggtitle(title)

plot_df(df_spike_count, exp_name)

In [ ]:
# Raw waveform over each channel
'''
WARNING: DO NOT USE LARGE DATASETS: MAXIMUM EXPERIMENT DURATION ~10MINS SAFELY

Needs breaking up into batch processing based on temporal codes to allow Finalspark server to not be overloaded
'''
raws = []
for i, row in tqdm(df_spike_event.iterrows(), total=df_spike_event.shape[0]):
    t = row['Time']
    t1 = t-timedelta(milliseconds=1)
    t2 = t+timedelta(milliseconds=2)
    raws.append(db.get_raw_spike(t1,t2, row['channel']))

def plot_raw_channel(channel: int):
    df_chan = df_spike_event[df_spike_event['channel']==channel]
    # Start with an empty plot
    p = ggplot()
    p += ggtitle(f'Channel {channel}')

    for i, row in df_chan.iterrows():
        df_raw = raws[i]
        if df_raw.shape[0] == 90:
            df_raw['Time [ms]'] = np.linspace(-1,2,90)
            p += geom_line(aes(x='Time [ms]', y='Amplitude'), data=df_raw)

    return p

channels = df_spike_event['channel'].unique()
list_plot = []
for i,chan in enumerate(channels):
    list_plot.append(plot_raw_channel(chan))

nbCol = 3
nbRow = len(list_plot)//nbCol
if len(list_plot)%nbCol != 0:
    nbRow += 1
gggrid(list_plot, ncol=nbCol) + ggsize(800, 300*nbRow)